# Preparing Your Data for Analysis
### Guided tutorial · Maize metabolomics case study

**Case study:** Cañas et al. (2017), *The Plant Cell*, [DOI: 10.1105/tpc.16.00613](https://doi.org/10.1105/tpc.16.00613). The experiment examined 19 maize inbred lines representing five genetic groups, with metabolic measurements at the vegetative (V) and 15-days-after-silking (15 DAS) stages.

**Your task:** Convert research data into an **analysis-ready table** without destroying the raw measurements or losing the experimental design.

**Learning outcomes**
1. Define the observational unit, variable types, and intended analysis.
2. Import a multi-sheet research workbook and inspect its layout.
3. Reshape or select the relevant table; identify samples, replicates, genotypes, population groups, and stages.
4. Check duplicates, data types, missingness, implausible values, and technical/batch concerns.
5. Make documented preprocessing decisions while avoiding data leakage.
6. Export cleaned data, a dictionary, and a quality-control report.

**Instructions:** Run cells in order. At each **Your turn** prompt, add a Markdown cell with your answer. Do not change the source data to make it fit an expected result.

**Important:** This is a *data-preparation* lesson, not a statistical-modeling lesson. PCA at the end is an exploratory QC check only.

## 1 · Know your data before opening it

The paper describes maize variation across genotypes, genetic groups, and development. Its supplementary **Dataset 1** contains raw replicate-level traits. Not every worksheet is necessarily formatted as a tidy sample-by-metabolite table; inspect the sheets before choosing how to parse them.

**Your turn 1**
- What is the experimental unit (plant, sample, genotype mean, or another unit)? How can you tell?
- Which columns are identifiers, factors, measurements, or metadata?
- Why is a genotype-group label *not* interchangeable with a genotype identifier?
- Why should you **not** combine V and 15-DAS observations without retaining stage information?

Source: [Research article and supplementary materials (PMC)](https://pmc.ncbi.nlm.nih.gov/articles/PMC5466022/).

In [ ]:
import io, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT = Path('maize_preparation_outputs')
OUT.mkdir(exist_ok=True)
print('pandas', pd.__version__)


## 2 · Download and inspect the actual supplementary workbook

When `LOAD_PUBLISHED_WORKBOOK=True`, the notebook attempts the publisher's public supplementary file. If the publisher blocks automated downloads, **download Dataset 1** from the article's Supplementary Materials and upload it in Colab. This fallback does not invent or replace the published data.

**Your turn 2:** Before importing a sheet, predict the dimensions and explain the difference between a spreadsheet header and an observation.

In [ ]:
# Set to True only when you are ready to inspect the published supplementary workbook.
LOAD_PUBLISHED_WORKBOOK = False
DATASET_1_URL = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5466022/bin/supp_tpc.16.00613_TPC2016-00613-LSBR2_Supplemental_Dataset_1.xlsx'
FILE = Path('maize_supplemental_dataset_1.xlsx')
xl = None
if LOAD_PUBLISHED_WORKBOOK:
    if not FILE.exists():
        try:
            import requests
            r=requests.get(DATASET_1_URL,timeout=25,headers={'User-Agent':'Mozilla/5.0'})
            r.raise_for_status()
            if r.content[:2] != b'PK': raise ValueError('Response is not an Excel workbook')
            FILE.write_bytes(r.content)
        except Exception as e:
            print('Automatic download unavailable:',type(e).__name__)
            print('Upload Dataset 1 using the Colab file sidebar, then rerun.')
            try:
                from google.colab import files
                uploaded=files.upload()
                if uploaded: FILE=Path(next(iter(uploaded)))
            except ImportError: pass
    if not FILE.exists(): raise FileNotFoundError('Upload published supplementary Dataset 1 (.xlsx).')
    xl=pd.ExcelFile(FILE)
    print('Worksheets:',xl.sheet_names)
else:
    print('Practice mode: supplementary workbook download skipped.')


In [ ]:
if xl is not None:
    for sheet in xl.sheet_names:
        preview=pd.read_excel(FILE,sheet_name=sheet,header=None,nrows=8)
        print(f'\n===== {sheet} =====')
        print('Preview shape:',preview.shape)
        display(preview)
else:
    print('To inspect published worksheets, set LOAD_PUBLISHED_WORKBOOK=True above.')


### Select a sheet and identify its structure

Change the following settings **after inspecting your workbook**. A sheet can contain titles, several blocks, header rows, and annotations. Do not assume row 1 is the variable-name row.

**Your turn 3:** Record worksheet name, true header row (0-based), what a row means, and any notes/footers to exclude.

In [ ]:
SHEET_NAME = xl.sheet_names[0] if xl is not None else None
HEADER_ROW = 0  # CHANGE: row with column names in published worksheet
raw_sheet = pd.read_excel(FILE,sheet_name=SHEET_NAME,header=HEADER_ROW) if xl is not None else None
if raw_sheet is not None: display(raw_sheet.head())


## 3 · Build the working table (do not overwrite raw data)

The supplementary workbook can contain *both measurements and summaries*. For this exercise, select only records that correspond to individual biological observations. If your selected sheet uses metabolites as rows and samples as columns, transpose it deliberately; if it includes several independent tables, extract the relevant block before proceeding.

For teaching purposes we also provide a **small explicitly simulated practice table** with 19 named example lines, two stages, three replicate measurements and six fictional metabolites. Use it only to learn the workflow if the supplementary workbook layout takes longer to decipher. **Do not report simulated values as Cañas et al. measurements.**

Set `DATA_MODE = 'practice'` to execute the tutorial end-to-end immediately; set `DATA_MODE = 'published'` after you have identified a tidy worksheet.

In [ ]:
DATA_MODE = 'practice'  # 'practice' or 'published'

if DATA_MODE == 'practice':
    rng = np.random.default_rng(42)
    lines = ['CML254','CML245','EM1201','ARGL256','P465P','FV252','MBS847','Mo17','ND283',
             'HP301','SA24','B73','C105','ND36','NYS302','FV2','Lo3','Lo32','F64']
    # For simulated examples ONLY: group assignments are illustrative, not source annotations.
    groups = ['T']*5 + ['CBD']*6 + ['SS']*2 + ['NF']*3 + ['EF']*3
    records=[]
    for gi,(line,group) in enumerate(zip(lines,groups)):
        for stage in ['V','15DAS']:
            for replicate in (1,2,3):
                values = rng.lognormal(mean=1.1 + gi*.025 + (stage=='15DAS')*.4,
                                       sigma=.3, size=6)
                rec = {'sample_id':f'{line}_{stage}_R{replicate}', 'genotype':line,
                       'population_group':group, 'stage':stage, 'replicate':replicate}
                rec.update({f'Met_{j+1}':v for j,v in enumerate(values)})
                records.append(rec)
    df = pd.DataFrame(records)
    df.loc[3,'Met_2'] = np.nan
    df.loc[12,'Met_4'] = np.nan
    df.loc[20,'Met_1'] = -1  # planted teaching QC problem
else:
    if raw_sheet is None: raise ValueError('Set LOAD_PUBLISHED_WORKBOOK=True in Section 2 and rerun.')
    df = raw_sheet.copy()

print('Mode:',DATA_MODE,'Shape:',df.shape)
display(df.head())


## 4 · Specify a data dictionary

The analysis-ready convention is **rows = individual measurements/samples** and **columns = variables**. Identify each column's role before applying numeric operations.

**Your turn 4:** Modify `ID_COLS` as required. Explain which columns represent categorical groupings and which represent measured metabolites.

In [ ]:
# EDIT for your published-data sheet. Leave the defaults for practice mode.
ID_COLS = ['sample_id','genotype','population_group','stage','replicate']
MEASUREMENT_COLS = [c for c in df.columns if c not in ID_COLS]

missing_ids = [c for c in ID_COLS if c not in df.columns]
if missing_ids:
    raise ValueError(f'Missing expected metadata columns: {missing_ids}. Inspect/reshape your published sheet first.')

dictionary = pd.DataFrame({'variable':df.columns,
                           'role':['metadata' if c in ID_COLS else 'measurement' for c in df.columns],
                           'dtype':[str(x) for x in df.dtypes],
                           'description':['TO COMPLETE' for _ in df.columns]})
display(dictionary)


## 5 · Audit observational units, duplicates and replication

Do **not** delete duplicate rows automatically. Duplicated values may be legitimate; duplicated *sample identifiers* may indicate a data-entry problem. Biological and technical replicates are not interchangeable.

**Your turn 5:** Identify the number of genotypes, stages, and independent observations. Are replication counts balanced? If not, should imbalance be corrected or documented?

In [ ]:
print('Rows:',len(df),'Columns:',len(df.columns))
print('Exact duplicate rows:',df.duplicated().sum())
print('Duplicated sample IDs:',df['sample_id'].duplicated().sum())
print('Genotypes:',df['genotype'].nunique(),'Stages:',df['stage'].unique())
rep_table = df.groupby(['genotype','stage'],dropna=False).size().unstack(fill_value=0)
display(rep_table)
assert not df['sample_id'].isna().any(), 'Missing sample IDs: investigate before proceeding'


## 6 · Check measurement types, missingness, and invalid values

First convert measurement-like columns to numbers without silently discarding original data. Non-numeric content becomes missing and must be audited. For concentration/abundance-like data, negative numbers are often invalid, but this depends on whether the measurements are raw, baseline-corrected, or transformed.

**Your turn 6:** Which missing values can be considered below detection? Which may be technical failures? What additional laboratory metadata would you need?

In [ ]:
working = df.copy(deep=True)
conversion_issues = {}
for c in MEASUREMENT_COLS:
    before = working[c].notna().sum()
    working[c] = pd.to_numeric(working[c], errors='coerce')
    conversion_issues[c] = before-working[c].notna().sum()

qc = pd.DataFrame({
    'missing_count':working[MEASUREMENT_COLS].isna().sum(),
    'missing_percent':working[MEASUREMENT_COLS].isna().mean().mul(100).round(2),
    'non_numeric_to_missing':pd.Series(conversion_issues),
    'negative_count':working[MEASUREMENT_COLS].lt(0).sum(),
    'zero_count':working[MEASUREMENT_COLS].eq(0).sum(),
    'n_unique':working[MEASUREMENT_COLS].nunique(dropna=True),
})
display(qc.sort_values('missing_percent',ascending=False).head(20))


In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4))
qc['missing_percent'].hist(ax=axes[0],bins=12)
axes[0].set(xlabel='Missing (%) per measurement',ylabel='Count of measurements',title='Missingness distribution')
working[MEASUREMENT_COLS].isna().sum(axis=1).hist(ax=axes[1],bins=12)
axes[1].set(xlabel='Missing values per sample',ylabel='Count of samples',title='Sample-level completeness')
plt.tight_layout();plt.show()


## 7 · Decide what to retain and what to flag

Use an explicit, documented policy; the cutoffs below are **teaching defaults, not rules from the paper**. Flag negative abundance values rather than silently converting them to zero. Retain sample metadata even if one metabolite is missing. Do not collapse genotype replicates into means at this stage.

**Your turn 7:** Propose reasonable retention cutoffs for *your* project. Explain a situation where 20% missingness would not warrant exclusion.

In [ ]:
MAX_FEATURE_MISSING = 0.20
MAX_SAMPLE_MISSING = 0.30

# For the practice table only, replace known planted invalid negative abundance with missing.
# For real data: verify measurement scale and instrument conventions before enabling.
INVALID_NEGATIVE_AS_MISSING = (DATA_MODE == 'practice')
if INVALID_NEGATIVE_AS_MISSING:
    working[MEASUREMENT_COLS] = working[MEASUREMENT_COLS].mask(working[MEASUREMENT_COLS] < 0)

sample_missing = working[MEASUREMENT_COLS].isna().mean(axis=1)
feature_missing = working[MEASUREMENT_COLS].isna().mean(axis=0)
keep_samples = sample_missing <= MAX_SAMPLE_MISSING
keep_features = feature_missing <= MAX_FEATURE_MISSING
retained_features = feature_missing.index[keep_features].tolist()
clean = working.loc[keep_samples,ID_COLS+retained_features].copy()
print('Sample rows retained:',len(clean),'/',len(working))
print('Measurement columns retained:',len(retained_features),'/',len(MEASUREMENT_COLS))
assert retained_features, 'All measurement features removed; revisit cutoffs/table structure'


## 8 · Examine distributions and understand transformations

Metabolite abundances often have right-skewed distributions. A log transform can improve interpretability and reduce spread, but zeros, negatives, and non-detects require principled handling. **Never automatically add a pseudocount without documenting its units and effect.** Here we compare values without transforming the saved raw-scale table.

**Your turn 8:** Which values are appropriate for log transformation? Why does scaling each feature alter distance-based comparisons?

In [ ]:
feature = retained_features[0]
x = clean[feature].dropna()
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(x,bins=18);axes[0].set(xlabel=feature,title='Original measurement scale')
if (x>0).all():
    axes[1].hist(np.log10(x),bins=18);axes[1].set(xlabel='log10('+feature+')',title='Exploratory log scale')
else:
    axes[1].text(.1,.5,'Nonpositive values present: log skipped',transform=axes[1].transAxes)
plt.tight_layout();plt.show()


## 9 · Keep biological factors visible in QC

Differences between stages and population groups may be biological signal—not nuisance variation. Before treating a cluster or apparent outlier as a technical problem, ask whether it aligns with genotype, stage, tissue, replication, or batch.

**Your turn 9:** Could population stratification influence which metabolites look associated with a phenotype? How would you avoid confusing population differences with technical artifacts?

In [ ]:
display(clean.groupby(['stage','population_group'],dropna=False).size().rename('n_samples').reset_index())
if 'stage' in clean.columns and len(clean['stage'].dropna().unique())>1:
    stages=[g[feature].dropna().values for _,g in clean.groupby('stage')]
    labels=[str(k) for k,_ in clean.groupby('stage')]
    plt.figure(figsize=(7,4));plt.boxplot(stages,tick_labels=labels)
    plt.title(f'{feature} by stage (QC, not inference)');plt.ylabel(feature);plt.show()


## 10 · Diagnose missingness before imputation

Adapted from the course [ImputationWorkflow_v1.ipynb](https://github.com/acesillinois/cpsc-ese-practicum/blob/main/ImputationWorkflow_v1.ipynb): **explore missing-value patterns → apply multivariate iterative imputation (MICE-style) → validate that distributions and relationships are preserved.**

The course workflow demonstrates `IterativeImputer(estimator=BayesianRidge())` for numeric variables. It is a *single iterative imputation using chained conditional models*; it does not, by itself, generate multiple imputed datasets or Rubin-pooled inference. Here we keep raw measured values, missingness indicators and imputations distinct.

Metabolomics caveat: before imputing, decide whether a missing signal represents technical failure, below-detection abundance, or biological absence. If likely below a limit of detection, the multivariate approach below may be inappropriate; discuss that case rather than automatically imputing.

**Your turn 10a:** Which samples/metabolites have the most missing measurements? Does missingness differ by genotype group or stage?

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_raw = clean[retained_features].copy().astype(float)
missing_mask = X_raw.isna()
print('Missing cells:', int(missing_mask.to_numpy().sum()), '/', X_raw.size)
print('Percent missing by stage:')
display(missing_mask.groupby(clean['stage']).mean().mean(axis=1).rename('missing_fraction'))
fig, ax = plt.subplots(figsize=(max(8, X_raw.shape[1]*.6),5))
ax.imshow(missing_mask.to_numpy(),aspect='auto',interpolation='nearest',cmap='Greys')
ax.set(title='Missingness map (dark = missing)',xlabel='Metabolites',ylabel='Samples')
ax.set_xticks(range(len(X_raw.columns)),labels=X_raw.columns,rotation=90)
plt.tight_layout();plt.show()

### 10b · Multivariate iterative imputation

We use the same `IterativeImputer` + `BayesianRidge` combination as your course example. Each metabolite is estimated using the other metabolites, rather than replacing every missing value by one fixed statistic. All-nonmissing features remain in the data; fully missing features cannot be recovered reliably and should be excluded or investigated.

**Important:** An imputed value is an *estimate*, not an observed concentration. This implementation is for exploratory data preparation. For prediction, fit imputation **only on training samples**, then transform held-out data. For inferential analyses, consider uncertainty-aware multiple imputation and the experimental design.

In [ ]:
# Retain a copy of the original measured values and their missingness mask.
if X_raw.isna().all().any():
    raise ValueError('A retained metabolite is entirely missing; review feature filtering.')

imputer = IterativeImputer(estimator=BayesianRidge(),max_iter=20,
                           tol=1e-3,random_state=42,skip_complete=False)
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw),index=X_raw.index,columns=X_raw.columns)

# Restore observed values exactly; imputer only supplies estimates where missing.
X_imputed = X_raw.where(~missing_mask, X_imputed)
assert not X_imputed.isna().any().any()
assert np.allclose(X_imputed.to_numpy()[~missing_mask.to_numpy()],
                   X_raw.to_numpy()[~missing_mask.to_numpy()])
print('Number of imputed cells:',int(missing_mask.to_numpy().sum()))
print('Observed measurements preserved exactly:',True)


### 10c · Validate imputation (do not judge by absence of NaNs alone)

Compare observed-versus-completed distributions, correlation structure, and missingness patterns. The imputed values should not silently eliminate stage or genotype differences. Because observed and imputed subsets can differ systematically, a close histogram match is not by itself proof of accuracy.

**Your turn 10b:** Which metabolites changed the most? Could group-specific imputation be preferable, or would group sizes become too small?

In [ ]:
from IPython.display import display
quality = pd.DataFrame({
    'n_missing': missing_mask.sum(),
    'mean_observed': X_raw.mean(),
    'mean_completed': X_imputed.mean(),
    'sd_observed': X_raw.std(),
    'sd_completed': X_imputed.std()
})
display(quality.round(3))

feature = X_raw.columns[0]
fig,axs = plt.subplots(1,2,figsize=(11,4))
axs[0].hist(X_raw[feature].dropna(),bins=15,alpha=.6,label='Observed')
axs[0].hist(X_imputed[feature],bins=15,alpha=.4,label='Completed')
axs[0].set(title=f'Distribution: {feature}',xlabel='Abundance');axs[0].legend()
C0=X_raw.corr(); C1=X_imputed.corr()
change=(C1-C0).abs().to_numpy()
axs[1].imshow(change,vmin=0,vmax=max(.01,np.nanmax(change)),cmap='viridis')
axs[1].set(title='Absolute change in pairwise correlation',xlabel='Metabolite index',ylabel='Metabolite index')
plt.tight_layout();plt.show()

# A diagnostic only: hold out a portion of observed cells, predict them,
# and calculate error. Do not interpret this as a predictive model test.
rng=np.random.default_rng(12)
positions=np.argwhere(~missing_mask.to_numpy())
n_hold=max(1,int(.10*len(positions)))
held=positions[rng.choice(len(positions),size=n_hold,replace=False)]
X_masked=X_raw.copy(); truth=[]
for row,col in held:
    truth.append(X_masked.iat[row,col]);X_masked.iat[row,col]=np.nan
try:
    check=IterativeImputer(estimator=BayesianRidge(),max_iter=20,random_state=12).fit_transform(X_masked)
    prediction=np.array([check[row,col] for row,col in held]); truth=np.array(truth)
    print('Masked-observation MAE:',round(float(np.mean(np.abs(prediction-truth))),4))
except ValueError as e:
    print('Masked-observation check unavailable:',e)


### 10d · Separate exploratory PCA from the original assay table

For PCA we standardize the **completed** numeric matrix, keeping the unimputed measurement table as the canonical record. PCA is an exploratory QC plot, not proof of population separation. If later modeling outcomes are used, fit the preprocessing steps on training data only.

In [ ]:
X_vis = StandardScaler().fit_transform(X_imputed)
if min(X_vis.shape) >= 2:
    xy=PCA(n_components=2).fit_transform(X_vis)
    plt.figure(figsize=(7,5))
    for stage in clean['stage'].dropna().unique():
        idx=(clean['stage']==stage).to_numpy()
        plt.scatter(xy[idx,0],xy[idx,1],label=str(stage),alpha=.8)
    plt.xlabel('PC1');plt.ylabel('PC2');plt.title('Exploratory PCA after iterative imputation')
    plt.legend();plt.show()


## 11 · Export reproducible outputs

Save **both** the clean unimputed table and the explicitly labeled iterative-imputed table, plus the original missing-value mask, QC summary, dictionary, and method log. This preserves the ability to repeat analyses without treating estimated values as directly measured.

**Your turn 11:** Could a classmate identify which values were observed and which were imputed? Explain why retaining the mask matters.

In [ ]:
clean.to_csv(OUT/'maize_analysis_ready_unimputed.csv',index=False)
completed=clean.copy()
completed.loc[:,X_imputed.columns]=X_imputed
completed.to_csv(OUT/'maize_analysis_ready_iterative_imputed.csv',index=False)
missing_mask.to_csv(OUT/'original_missing_mask.csv',index=False)
dictionary.to_csv(OUT/'variable_dictionary_to_complete.csv',index=False)
qc.to_csv(OUT/'measurement_qc.csv',index_label='measurement')
log = {'source':'Canas et al. 2017 supplementary Dataset 1' if DATA_MODE=='published' else 'SIMULATED PRACTICE DATA — NOT PUBLISHED MEASUREMENTS',
       'mode':DATA_MODE,'sheet':SHEET_NAME if DATA_MODE=='published' else None,
       'header_row':HEADER_ROW if DATA_MODE=='published' else None,
       'observation_unit':'Individual sample/replicate (verify for published sheet)',
       'max_feature_missing_fraction':MAX_FEATURE_MISSING,
       'max_sample_missing_fraction':MAX_SAMPLE_MISSING,
       'negative_as_missing':INVALID_NEGATIVE_AS_MISSING,
       'imputation_method':'sklearn IterativeImputer with BayesianRidge (single iterative imputation)',
       'imputation_reference':'https://github.com/acesillinois/cpsc-ese-practicum/blob/main/ImputationWorkflow_v1.ipynb',
       'imputed_cells':int(missing_mask.to_numpy().sum()),
       'standardized_export':False,
       'retained_sample_count':len(clean),'retained_measurement_count':len(retained_features)}
(OUT/'preprocessing_log.json').write_text(json.dumps(log,indent=2))
print('Saved outputs:');print(*sorted(p.name for p in OUT.iterdir()),sep='\n - ')


## 12 · Submission and reflection

Submit **(a)** completed notebook, **(b)** unimputed table, **(c)** iteratively imputed table, **(d)** original missingness mask, **(e)** variable dictionary and QC summary, and **(f)** preprocessing log.

**Short response (200–300 words)**
1. What is the observational unit and which factors identify a sample?
2. Is missingness random with respect to metabolites, genotype groups, or developmental stages?
3. Why is iterative multivariate imputation different from substituting a mean or median?
4. What do distribution and correlation diagnostics suggest about the imputation?
5. How could imputation affect within-genotype heterogeneity and the classification of stable vs plastic metabolites?
6. What would you need to confirm before interpreting a missing signal as a biologically absent metabolite?

**Completion checklist**
- [ ] Original source untouched and mode identified as published or simulated.
- [ ] Sample identifiers and biological factors preserved.
- [ ] Missingness visualized and quantified *before* imputation.
- [ ] Iterative imputation performed on numeric metabolite variables only.
- [ ] Observed values and missing-value mask preserved.
- [ ] Imputation evaluated against distributions and correlations.
- [ ] Imputed table kept separate from unimputed observations.
- [ ] Preprocessing parameters documented; train/test leakage avoided if used for prediction.

**Instructor note:** In published-data mode, students must identify and restructure the source worksheets and verify the actual genetic-group metadata. The simulated table is a training scaffold only.